In [1]:
import pandas as pd
import danda # noqa: F401  # registers the pandas accessor
from danda.report_renderer import ReportRenderer

## Dataset

This dataset is designed to demonstrate outlier detection and handling techniques. It contains four numeric columns with values that are primarily concentrated within a normal range, together with several intentionally inserted extreme values.

The dataset includes:

* **30 rows**
* **4 numeric columns**
* **1 categorical column**

### Columns

| Column   | Description                   |
| -------- | ----------------------------- |
| `age`    | Employee age in years.        |
| `salary` | Annual salary.                |
| `height` | Height in centimeters.        |
| `score`  | Performance score.            |
| `city`   | City name (categorical data). |

### Outliers

Each numeric column contains a mixture of:

* **Typical values** representing the majority of the observations.
* **Borderline values** that lie near the outlier threshold and may or may not be classified as outliers depending on the detection method and configuration.
* **Extreme values** that are expected to be detected as outliers.

This combination makes the dataset suitable for comparing different outlier detection methods, such as IQR, Z-score, and Modified Z-score.

### Purpose

The dataset can be used to demonstrate how to:

* Analyze numeric columns for outliers.
* Compare different outlier detection methods.
* Visualize outlier distributions.
* Remove rows containing outliers.
* Replace outliers with missing values (`NaN`).
* Clip outliers to the nearest acceptable boundary.
* Impute outliers after replacing them with missing values.

Because the dataset contains multiple numeric columns with different value ranges, it provides realistic examples of how the same detection method can produce different results depending on the underlying distribution of each variable.


In [2]:
df = pd.DataFrame({
    "age": [
        24, 25, 26, 27, 28, 29, 30, 31, 32, 33,
        24, 25, 26, 27, 28, 29, 30, 31, 32, 33,
        25, 26, 27, 28,
        45,   # borderline
        60,   # borderline
        90,   # high but may/may not be flagged
        120,  # definite outlier
        -20,  # low but may not be flagged
        -50   # definite low outlier
    ],

    "salary": [
        48000, 49500, 50000, 50500, 51000, 51500, 52000, 52500, 53000, 53500,
        49000, 50000, 51000, 52000, 53000, 54000, 55000, 56000, 57000, 58000,
        49500, 50500, 51500, 52500,
        65000,    # borderline
        90000,    # high but probably not
        120000,   # possible outlier
        500000,   # definite outlier
        1000,     # low but may not
        -100000   # definite low outlier
    ],

    "height": [
        168, 169, 170, 171, 172, 173, 174, 170, 171, 172,
        169, 170, 171, 172, 173, 174, 175, 171, 172, 173,
        170, 171, 172, 173,
        180,   # borderline
        190,   # high but probably not
        205,   # possible outlier
        250,   # definite outlier
        80,    # definite outlier
        40     # definite outlier
    ],

    "score": [
        85, 86, 87, 88, 89, 90, 91, 92, 93, 94,
        86, 87, 88, 89, 90, 91, 92, 93, 94, 95,
        87, 88, 89, 90,
        96,    # borderline
        98,    # high but probably not
        100,   # high
        120,   # definite outlier
        0,     # definite outlier
        -20    # definite outlier
    ],

    "city": [
        "London", "Paris", "Rome", "Berlin", "Madrid",
        "Lisbon", "Vienna", "Prague", "Warsaw", "Dublin",
        "Oslo", "Stockholm", "Helsinki", "Copenhagen", "Brussels",
        "Amsterdam", "Zurich", "Geneva", "Athens", "Budapest",
        "Sofia", "Zagreb", "Belgrade", "Ljubljana",
        "Sarajevo", "Skopje", "Tallinn",
        "Tokyo", "Sydney", "New York"
    ]
})

df

,age,salary,height,score,city
0,24,48000,168,85,London
1,25,49500,169,86,Paris
2,26,50000,170,87,Rome
3,27,50500,171,88,Berlin
4,28,51000,172,89,Madrid
5,29,51500,173,90,Lisbon
6,30,52000,174,91,Vienna
7,31,52500,170,92,Prague
8,32,53000,171,93,Warsaw
9,33,53500,172,94,Dublin


Outlier analysis report

The Outlier Report summarizes the outliers detected in each numeric column. For every column, it shows the detection method, the number of outliers found, the calculated outlier boundaries, an ASCII visualization of the value distribution, and example rows containing outliers.

The report is intended to help you understand why values were classified as outliers before deciding whether to remove, replace, or clip them.

Legend

The ASCII chart uses the following symbols:

Symbol	Meaning
<	The lower bound lies below the observed values.
>	The upper bound lies above the observed values.
=	Values within the acceptable range.
-	Outlier region.
`	`	Outlier threshold (lower or upper bound).
Report structure

Each numeric column contains the following information.

Method

The outlier detection algorithm used.

Example:

Method: MODIFIED-ZSCORE (×3.0)

The threshold shown in parentheses is the configuration used to determine whether a value is considered an outlier.

Outliers

The number and percentage of observations classified as outliers.

Example:

Outliers: 2 of 30 (6.67%)
Outlier interval

The smallest and largest outlier values detected.

Example:

Outlier Interval: -20 to 0

If only one outlier is detected, both values are identical.

Outlier Interval: 500000 to 500000
Bounds

The calculated lower and upper limits for acceptable values.

Bounds:
Lower: 9.82
Upper: 170.18

Values outside these bounds are classified as outliers.

Distribution graph

An ASCII visualization shows where the observed values lie relative to the calculated bounds.

For example:

-20                                            120
----------|======================================>
         LB

In this example:

values to the left of the lower bound are outliers;
the vertical bar (|) marks the lower threshold;
the equal signs (=) represent values within the acceptable range;
the arrow (>) indicates that the upper bound lies beyond the observed maximum.

Another example:

-50                                            120
<=============================================|---
                                             UB

Here, the lower bound is below the minimum observed value, while values beyond the upper threshold are classified as outliers.

Examples

The report lists example rows containing detected outliers.

Examples:
- Row 27: 500000

These examples help locate suspicious observations within the original DataFrame.

Notes

When a relatively large proportion of observations are classified as outliers, the report displays an informational note such as:

A relatively large proportion of values were classified as outliers.
This may indicate a skewed distribution rather than data quality issues.

This serves as a reminder that statistical outliers are not necessarily erroneous values. A high outlier rate may indicate that the data are naturally skewed, contain multiple populations, or require a different detection method or threshold.

Interpreting the example report

In this example:

score contains 2 low outliers (0 and -20).
age contains 1 high outlier (120).
height contains 1 low outlier (40).
salary contains 1 high outlier (500000).

Some extreme-looking values, such as -50 for age, 250 for height, -100000 for salary, or 120 for score, are not classified as outliers because they remain within the bounds calculated by the Modified Z-score method for those columns. This illustrates that outlier detection depends on the overall distribution of the data rather than on intuition alone.

In [3]:
report = df.dg.analyze().dg.report

In [4]:
renderer = ReportRenderer()
print(renderer.render(report))

Danda Report

Analyze
-------

ColumnSummary
    Column Summary:
    
    age
    Type: int64
    Missing: 0 (0%)
    Unique: 16
    
    salary
    Type: int64
    Missing: 0 (0%)
    Unique: 22
    
    height
    Type: int64
    Missing: 0 (0%)
    Unique: 14
    
    score
    Type: int64
    Missing: 0 (0%)
    Unique: 17
    
    city
    Type: str
    Missing: 0 (0%)
    Unique: 30

MissingValuesSummaryPlugin
    Missing Value Summary
    Rows: 30
    Columns: 5
    
    Missing cells: 0 (0.0%)
    Columns with missing: 0
    Rows with missing: 0
    Complete rows: 30 (100.0%)

OutlierReport
    Outliers detected:
    
    Legend:
    <  Bound lies below observed values
    >  Bound lies above observed values
    =  Values within normal bounds
    -  Outlier region
    |  Outlier threshold
    
    score
    Method: MODIFIED-ZSCORE (×3.0)
    Outliers: 2 of 30 (6.67%)
    Outlier Interval: -20 to 0
    
    Bounds:
    Lower: 9.82
    Upper: 170.18
    
    -20                  

Replace outliers with missing values

Instead of removing entire rows, you can replace outlier values with missing values (NaN).

This approach preserves all observations while marking only the outlier values as missing. The resulting missing values can then be handled using df.dg.impute().

noout = df.dg.action.handle_outliers(
    columns=["age"],
    strategy="nan",
)

In this example:

Only the age column is processed.
Values identified as outliers are replaced with NaN.
All non-outlier values remain unchanged.
No rows are removed from the DataFrame.

This strategy is particularly useful when outliers are believed to be erroneous or unrepresentative, but the remaining data in the affected rows should be retained.

A common workflow is:

df = (
    df
    .dg.action.handle_outliers(
        columns=["age"],
        strategy="nan",
    )
    .dg.impute()
)

Here, outliers are first converted to missing values and then filled using the configured imputation strategy (for example, the median for numeric columns). This often produces more representative values than clipping outliers to the nearest boundary.

In [5]:
# age[27] = nan
noout = df.dg.action.handle_outliers(
    columns=["age"],
    strategy="nan",
)
noout

,age,salary,height,score,city
0,24.0,48000,168,85,London
1,25.0,49500,169,86,Paris
2,26.0,50000,170,87,Rome
3,27.0,50500,171,88,Berlin
4,28.0,51000,172,89,Madrid
5,29.0,51500,173,90,Lisbon
6,30.0,52000,174,91,Vienna
7,31.0,52500,170,92,Prague
8,32.0,53000,171,93,Warsaw
9,33.0,53500,172,94,Dublin


In [6]:
# before clipping age[27] has a value of 120 age uper is 110.xx
# age[27] = 110.xx
noout = df.dg.action.handle_outliers(
    columns=["age"],
    strategy="clip",
)
noout

,age,salary,height,score,city
0,24.000000,48000,168,85,London
1,25.000000,49500,169,86,Paris
2,26.000000,50000,170,87,Rome
3,27.000000,50500,171,88,Berlin
4,28.000000,51000,172,89,Madrid
5,29.000000,51500,173,90,Lisbon
6,30.000000,52000,174,91,Vienna
7,31.000000,52500,170,92,Prague
8,32.000000,53000,171,93,Warsaw
9,33.000000,53500,172,94,Dublin


## Identify outliers with a mask

Use the **`mask`** strategy to identify which values are classified as outliers without modifying the DataFrame.

The returned DataFrame contains boolean values, where `True` indicates an outlier and `False` indicates a non-outlier.

```python
mask_df = df.dg.action.handle_outliers(
    columns=["age"],
    strategy="mask",
)
```

In this example:

* Only the **`age`** column is analyzed.
* The original DataFrame is **not** modified.
* The result is a boolean mask that can be used for inspection, filtering, or debugging.

For the example dataset, the mask indicates that row **27** contains an outlier:

```text
27    True
```

All other rows contain `False`.

The `mask` strategy is useful for:

* Verifying which values will be treated as outliers.
* Inspecting the results of different detection methods or thresholds.
* Building custom outlier handling workflows.
* Diagnosing the effect of configuration changes without altering the data.

Because no values are removed or replaced, this strategy is ideal for exploratory analysis and validation before applying a modification strategy such as `remove`, `nan`, or `clip`.


In [7]:
# mask can be used for diagnostics it will not chNGE ANYTHING
# This will sho that node 27 is true
mask_df = df.dg.action.handle_outliers(
    columns=["age"],
    strategy="mask",
)
mask_df

,age
0,False
1,False
2,False
3,False
4,False
5,False
6,False
7,False
8,False
9,False


 ##   salary
    Method: MODIFIED-ZSCORE (×3.0)
    Outliers: 1 of 30 (3.33%)
    Outlier Interval: 500,000 to 500,000

    Bounds:
    Lower: -214,189.11
    Upper: 318,189.11

    -100000                                     500000
    <=================================|---------------

In [11]:
df.loc[df["salary"]>318189]

,age,salary,height,score,city
27,120,500000,250,120,Tokyo


In [15]:
# LET US LOOK AT salary
# the outlier is at row 27
salary_df = df.dg.action.handle_outliers(
    columns=["salary"],
    strategy="nan",
)
salary_df

,age,salary,height,score,city
0,24,48000.0,168,85,London
1,25,49500.0,169,86,Paris
2,26,50000.0,170,87,Rome
3,27,50500.0,171,88,Berlin
4,28,51000.0,172,89,Madrid
5,29,51500.0,173,90,Lisbon
6,30,52000.0,174,91,Vienna
7,31,52500.0,170,92,Prague
8,32,53000.0,171,93,Warsaw
9,33,53500.0,172,94,Dublin
